In [1]:

from collections import Counter
from functools import lru_cache

import math
import re
import sage.libs.lrcalc.lrcalc as lrcalc
from sage.all import *
import sage.combinat.partitions as partitions

from sage.misc.latex import LatexExpr
from sage.combinat.composition import Compositions

all_partitions = partitions.ZS1_iterator
@lru_cache(maxsize=None)
def _lrcoeff_cached(outer, left, right):
    return lrcalc.lrcoef_unsafe(outer, left, right)

def lrcoeff(outer, left, right):
    """Memoized ordinary LR coefficient; accepts partition tuples and lists."""
    return _lrcoeff_cached(tuple(outer), tuple(left), tuple(right))

def clear_lr_caches():
    """Clear the ordinary and three-factor LR caches."""
    _lrcoeff_cached.cache_clear()
    lrcoef3_cached.cache_clear()
    _subset_partitions_cached.cache_clear()





In [2]:
def degree(partition):
    return sum(partition)

def compare_by_degree(p1, p2):
    d1, d2 = degree(p1), degree(p2)
    if d1 < d2:
        return -1
    elif d1 > d2:
        return 1
    return 0


In [3]:
# Example (uncomment to run):
# degree((3,))


In [4]:

def generate_partitions(n, max_part=None):
    """Generate all partitions of n."""
    if n == 0:
        yield ()
    else:
        if max_part is None or max_part > n:
            max_part = n
        for first in range(max_part, 0, -1):
            for rest in generate_partitions(n - first, first):
                yield (first,) + rest

# Example (uncomment to run):
# p = (3, 1)
# parts = list(generate_partitions(5))
# print(parts)




In [5]:
@lru_cache(maxsize=None)
def _subset_partitions_cached(partition):
    """Return all Young-diagram subpartitions of `partition`, including ()."""
    result = []

    def visit(i, previous, prefix):
        if i == len(partition):
            result.append(tuple(prefix))
            return
        for value in range(min(previous, partition[i]), -1, -1):
            if value == 0:
                result.append(tuple(prefix))
                break  # all remaining parts must be zero
            prefix.append(value)
            visit(i + 1, value, prefix)
            prefix.pop()

    visit(0, partition[0] if partition else 0, [])
    return tuple(result)


def subset_partitions(partition):
    """Return actual subpartitions; cached internally, with a fresh list result."""
    return list(_subset_partitions_cached(tuple(partition)))


# Example: subset_partitions((3,1))

In [6]:
# Example (uncomment to run):

# print("Degree of p:", degree(p))

# q = (1, 1)
# print("Compare p and q:", compare_by_degree(p, q))  # 0 means equal

# subset = subset_partitions(p)
# print(f"All partitions with degree ≤ {degree(p)}:")
# print(subset)


In [7]:
# Example (uncomment to run):

# print(lrcoeff((4,), (2,), (1,1)))
# print(lrcalc.lrcoef((1,), (1,), (2,)))

# tests 2




In [8]:
def solveTwo(k, k1, k2, k3, k4):

    if k2 > k1 or k4 > k3:
        return []

    solutions = []


    #print(f'Start of iteration for Delta (from 0 to k):\n')
    for x1 in range(k + 1):
        #print(f'======= The {x1}-th ITERATION FOR DELTA_1: ========\n')
        #print(f'    Value of delta1 = {x1}')

        # max_x2 = math.floor((k - x1)) I dont think i need to run through since we have no p,q and x_2 just equals k - x_1
        # range_Max = math.ceil((max_x2)/2 + 1) Leave it just in case
        #print(f'    Maximum Gamma = {max_x2}\n')
        #print(f'    Start of iteration for Gamma (from 0 to ceil(floor(k - delta)/2) + 1): \n' )

        # that means we can destroy the for loop:

        x2 = k - x1
        
        #for x2 in range(range_Max):
            #print(f'------- {x2}-th iteration of Gamma: --------\n')
       # print(f'        Value of delta2 = {x2}')


        #print(f'{k1-k2}={k3-k4}={k}')
        if k == k1-k2 == k3-k4 :

                sol1 = {
                "deg γ1": x1,
                "deg γ2": x2,
                "deg λ'": k2,
                "deg μ'": k4
                
                }
                #print(f'       Solution is {sol1}')
                solutions.append(sol1)


        
            
        if (x1 + x2 != k) or (x1 + x2 != k1 - k2) or (x1 + x2 != k3 - k4):
                #print('Skipped')
                continue
    return solutions


In [9]:
# Example: Correct Checked by Hand

# print(solveTwo(3, 4, 1, 4, 1))


In [10]:
def restrict_partitions(part1_list, part2_list, d_lam):
    """
    Filters two lists of partitions (part1_list, part2_list) so that
    only pairs (a, b) satisfy |a| + |b| = |lam|, where |.| is the degree (sum of parts).

    Returns:
        (filtered_part1, filtered_part2)
        where both are lists of partitions that can potentially combine to lam.
    """

    filtered_part1 = []
    filtered_part2 = []

    for a in part1_list:
        for b in part2_list:
            if sum(a) + sum(b) == d_lam:
                filtered_part1.append(a)
                filtered_part2.append(b)

    return filtered_part1, filtered_part2


In [11]:


@lru_cache(maxsize=None)
def lrcoef3_cached(mu1, mu2, mu3, lam):
    """Three-factor LR coefficient, using only target-contained intermediates."""
    tot = degree(mu1) + degree(mu2)
    if tot + degree(mu3) != degree(lam):
        return 0
    result = 0
    for nu in subset_partitions(lam):
        if degree(nu) == tot:
            result += lrcoeff(nu, mu1, mu2) * lrcoeff(lam, nu, mu3)
    return result


def lrcoef3(mu1, mu2, mu3, lam):
    """Preserved four-argument interface, accepting tuples or lists."""
    return lrcoef3_cached(tuple(mu1), tuple(mu2), tuple(mu3), tuple(lam))


In [12]:
# # print(f'{lrcoef4((1,), (), (), (1,), (1,1))} \n done! \n')
# print(f'{lrcoef4((), (), (), (1,), (1,))} \n done! \n')
print(f'{lrcoef3((1,1,),(1,1),(1,1,), (3,3,))}') 






1


In [13]:
def ones_partition(x):
    """Return the partition (1, 1, ..., 1) with x ones."""
    if x < 0:
        raise ValueError("x must be nonnegative")
    return tuple(1 for _ in range(x))


In [14]:
def CalcSoc(k, lam, lamP, mu, muP, verbose=False):
    """
    k1 = |lam|,   k3 = |mu|
    """

    k1 = degree(lam)
    k2 = degree(lamP)
    k3 = degree(mu)
    k4 = degree(muP)

    #print (f'Values of the degrees of: \n lambda, delta lambda Prime, mu prime, deg mu Prime, {k1, k2, k3, k4} \n')
    
    L = solveTwo(k, k1, k2, k3, k4)

    #print(f'Solution degrees for Delta1, delta2  are:\n')
    #print(f'    {L}\n\n')

    #print('++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++\n')
    #print('++++++++++++++++++ START of Total Summation ++++++++++++++++++\n')
    #print('++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++\n')

    
    tot_sum = 0
    nicegammas = []

    for sol in L:
        #print(f'For {sol}: \n Lists of possible partitions for:\n')

        d1_list = list(generate_partitions(sol["deg γ1"]))
        d2_list = list(generate_partitions(sol["deg γ2"]))
        #print(f'    Delta: \n {d1_list}')
        #print(f'    Gamma: \n {d2_list}')


        #print("\n=========== START OF LOOP IN DELTA FOR EACH PRODUCT ========\n")
        for d in d1_list:
            #print(f'-------------- START OF LOOP IN GAMMA --------------')
            for g in d2_list:
                #print("\n=========== START OF LOOP FOR EACH PRODUCT ========\n")
                #print(f'solution partitions: d:{d}, g: {g}')


                # first factor uses the *full* partition lam
                term1 = lrcoef3(d, g, lamP, lam)
                #print(f'term (N^lam) = {term1}')

                # second factor uses the *full* partition mu
                term2 = lrcoef3(d, g, muP, mu)
                #print(f'term (N^mu) = {term2}')

                N = term1 * term2
                
                if N != 0:
                    nicegammas.append((d,g))
                tot_sum += term1 * term2
                #print(f'sum is currently: {tot_sum}')
            #print('------------------------- END ---------------------------\n')
        
        #print('============================ END ==================================')

    if verbose and len(nicegammas) != 0:
        print(f'The list of gamma1,gamma2 for lamba prime = {lamP} and muprime = {muP} is \n {nicegammas}')
    return tot_sum


In [15]:
def Master1(k, lam, lamP, mu, muP, ex_string = ''):



    print(f'------ Calculation of suitable partitions delta, gamma, 1^p, 1^q ------ \n')
    print(solveTwo(k, degree(lam), degree(lamP), degree(mu), degree(muP)))
    print(CalcSoc(k, lam, lamP, mu, muP))


# My Example with Master

# Master1(3, (1, 2), (1, 1), (2,), (), 'My Example')
# My Example



# Ex 1

# Master1(1, (6,), (4,2), (1,1,1,1,1), (5,), 'Example 1')


# Ex 2

# Master1(1, (1,), (), (1,), (), 'Example 2')


# Ex 3

# Master1(2, (1,), (), (1,), ())

# Ex 4

# Master1(2, (1,1), (), (1,), (), 'Example 4') # also works for k =3

#Ex 5

# Master1(2, (1,1), (1,), (1,), (), 'Example 5') # CHECKS FINE!!!

# Master1(3, (1,1), (), (1,), (), 'Example 6')


# new example

# Master1(2, (1,), (), (1,), (1,), "new example")

#Master1(4, (1,1), (), (1,1), ())

# Big exampple:


# Example (uncomment to run):
# k = 13
# lam = (5,3,2,2,)
# mu = (4,4,4,)
# lamP = ()
# muP = ()
# Master1(k, lam, lamP, mu, muP)



# Graphing with Latex

First create string for a row: the direct sum of all possible subpartition combinations


In [16]:


def DirectSumMethod(k, lam, mu):
    muP_list = subset_partitions(mu)
    lamP_list = subset_partitions(lam)

    terms = []

    for lamP in lamP_list:
        for muP in muP_list:
            S = CalcSoc(k, lam, lamP, mu, muP)
            # print(f'lP = {lamP}, mP = {muP}, k = {k}      S = {S}')

            #print(f'S is {S}')
            if S != 0:
                # print("!!\n")
                lamP_str = r"\varnothing" if lamP == () else str(lamP)
                muP_str  = r"\varnothing" if muP == () else str(muP)

                exponent = f"{{{lamP_str}, {muP_str}}}"

                if S == 1:
                    term = f"V^{exponent} \\oplus "
                else:
                    term = f"{S} V^{exponent} \\oplus "
                
                terms.append(term.removesuffix(r"\oplus ").rstrip())

    return r" \oplus ".join(terms)

# Example Use to produce a term in the table
# print(DirectSumMethod(12, (5,3,2,2,), (4,4,4,)))
# print(DirectSumMethod(, (2,1,1,1,1,), (3,2,)))


## Make table by iterating rows

2 methods: one for string, and one with file managemenent generates a tex file with the contents.

In [17]:

def MakeTable(k, lam, mu):
    print(f"Table for $\\lambda da = {lam}$, $\\mu = {mu}, k = {k}$:\n")

    print("\\begin{table}[h]")
    print("\\centering")
    print("\\begin{tabular}{|c|}")
    print("\\hline")

    for i in range(2*k):
        row = DirectSumMethod(k-i, lam, mu)

        if row == f"":
            continue
        #if row == "":
            #row = r"$0$"        # optional: ensure the table cell isn't empty
        print(f"${row}$ \\\\ \\hline")
        

    print(f' $V^{{{lam}, {mu}}}$\\\\ \\hline')
    print("\\end{tabular}")
    print("\\end{table}")



def MakeTable_tex(lam, mu, filename,):
    # Build the LaTeX content
    
    tex = []
    tex1 = []

    tex.append(f"\\renewcommand{{\\arraystretch}}{{1.6}} \n")
    tex.append("\\begin{table}[H]\n")
    tex.append("\\centering\n")
    tex.append(f"\\caption{{Table for $\\lambda = {lam}$, $\\mu = {mu}$:}}\n")
    tex.append("\\begin{tabular}{|>{\\centering\\arraybackslash}p{15cm}|}\n")
    tex.append("\\hline\n")


    k = 0
    while True:
        row = DirectSumMethod(k, lam, mu)
        if row == f"":
            break
        row = re.sub(r"\((\d+),\)", r"(\1)", row)
        tex1.append(f" $k = {k}: \quad {row}$ \\\\ \\hline\n")
        k += 1
    
    tex1.reverse()
    tex += tex1
    tex.append("\\end{tabular}\n")
    tex.append("\\end{table}\n")

    # Join into a single string

    tex_string = "".join(tex)

    # Write to file
    with open(filename, "a") as f:
        f.write(tex_string)
        f.write("\n%---------------------\n")

    print(f"LaTeX table written to {filename}")
    return tex_string









In [18]:
# Example use
#MakeTable((1,1), (2,), "../hoyt_tables/t1.tex")
# MakeTable(2,(5,3,2,2), (4,4,4,))

# Main Operator to make table


In [19]:
filename = "./tex_tables/t_all.tex"

def generateTables(n, m, file =filename):
    lam_list = list(generate_partitions(n))
    mu_list = list(generate_partitions(m))


    print(f'For $|\\lambda| = {n}$ and $|\\mu| = {m}$:\n')
    print(f'')
        # Write to file



    print(f"LaTeX table written to {filename}")
    for lam in lam_list:
        for mu in mu_list:
            MakeTable_tex(lam, mu, file)

        




def le_generateTables(max_m, max_n):


    for i in range(max_n + 1):
        for k in range(max_m + 1):
            
            generateTables(i, k, f"../hoyt_tables/t_all.tex")



In [20]:
# USAGE IS HERE ----------------

generateTables(7, 8, "../hoyt_tables/t1.tex")

# le_generateTables(3, 3)

For $|\lambda| = 7$ and $|\mu| = 8$:


LaTeX table written to ./tex_tables/t_all.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table written to ../hoyt_tables/t1.tex
LaTeX table wri

## Validation and caching
Uncomment the examples below to verify the preserved interfaces and inspect cache hits. The original `CalcSoc` summation and `MakeTable_tex` stopping behavior are unchanged.


In [21]:
# Small checks (uncomment to run in SageMath):
# assert set(subset_partitions((3, 1))) == {(), (1,), (2,), (3,), (1,1), (2,1), (3,1)}
# assert lrcoef3((1,), (1,), (), (2,)) == 1
# assert lrcoef3((1,), (1,), (), (1,1)) == 1
# print("Ordinary LR cache:", _lrcoeff_cached.cache_info())
# print("Three-factor LR cache:", lrcoef3_cached.cache_info())
# clear_lr_caches()


## Large-scale regression verification (SageMath; no pandas)

This section reuses the original `hoyt.ipynb` definitions as an independent reference. It tests many **complete `CalcSoc` coefficients**, distributed across source partition pairs and degrees `k`, plus separate `lrcoef3` coefficients. Nothing is printed as a full table: only an aggregate report, degree-level summary, and a few failures. All generated test cases and discrepancies can optionally be saved to CSV. Run the setup cell, then the test cell, then the report cell. 

The original implementation is substantially slower; start with the defaults and increase `MAX_TESTS`/`MAX_DEGREE` when appropriate. Matching results validate only the tested inputs against the old code, not mathematical correctness of both implementations.

In [22]:
import csv, io, json, random, time, builtins
from collections import defaultdict
from contextlib import redirect_stdout
from pathlib import Path
from html import escape
from IPython.display import display, HTML

# Read the original notebook, without running its examples or table-producing cells.
reference_path = Path("hoyt.ipynb")
if not reference_path.is_file():
    raise FileNotFoundError("Place the original hoyt.ipynb beside this notebook.")
reference_nb = json.loads(reference_path.read_text(encoding="utf-8"))
reference = {"__name__": "hoyt_original_reference"}
for cell_no in (0, 1, 3, 4, 8, 11, 14):
    source = "".join(reference_nb["cells"][cell_no]["source"])
    if cell_no == 3:
        source = source.split("\np = (3, 1)")[0]
    if cell_no == 4:
        source = source.split("\nsubset_partitions((3,1))")[0]
    with redirect_stdout(io.StringIO()):
        exec(compile(source, f"hoyt.ipynb:cell_{cell_no}", "exec"), reference)
old_CalcSoc = reference["CalcSoc"]
old_lrcoef3 = reference["lrcoef3"]

# Configuration — increase these after an initial successful run.
MAX_DEGREE = 7       # source partitions have degree 1..MAX_DEGREE
MAX_PAIRS = 180      # randomly chosen ordered (lambda, mu) pairs
MAX_TESTS = 3000     # maximum number of CalcSoc coefficient comparisons
MAX_LR3_TESTS = 400  # independently compare this many 3-factor coefficients
SEED = 20261028     # ensures reproducible samples
PROGRESS_EVERY = 250
SAVE_ALL_CSV = False # True to retain every CalcSoc comparison as a CSV

# Generate eligible complete CalcSoc cases; randomize before capping to avoid
# systematically testing only the smallest degrees.
rng = random.Random(int(SEED))
source_parts = [tuple(p) for n in range(1, MAX_DEGREE + 1)
                for p in generate_partitions(n)]
source_pairs = [(a, b) for a in source_parts for b in source_parts]
rng.shuffle(source_pairs)
source_pairs = source_pairs[:MAX_PAIRS]

# Make a test pool without creating huge tables, retaining only degree-valid primes.
case_pool = []
for lam, mu in source_pairs:
    lp_by_degree, mp_by_degree = defaultdict(list), defaultdict(list)
    for p in subset_partitions(lam):
        lp_by_degree[builtins.sum(p)].append(tuple(p))
    for p in subset_partitions(mu):
        mp_by_degree[builtins.sum(p)].append(tuple(p))
    for k in range(min(degree(lam), degree(mu)) + 1):
        for lamP in lp_by_degree[degree(lam) - k]:
            for muP in mp_by_degree[degree(mu) - k]:
                case_pool.append((k, lam, lamP, mu, muP))
rng.shuffle(case_pool)
test_cases = case_pool[:MAX_TESTS]
print(f"Ready: {len(test_cases):,} CalcSoc cases across {len(source_pairs):,} source pairs ")
print(f"(pool size: {len(case_pool):,}; degrees 1..{MAX_DEGREE}; seed {SEED}).")


FileNotFoundError: Place the original hoyt.ipynb beside this notebook.

In [23]:
# Compare original and modified numerical outputs, collect timing, preserve failures.
comparison_rows = []
failures = []
by_k = defaultdict(lambda: {"tests": 0, "passed": 0, "old_sec": 0.0, "new_sec": 0.0, "nonzero": 0})
by_pair = defaultdict(lambda: {"tests": 0, "passed": 0})
started = time.perf_counter()
for i, args in enumerate(test_cases, 1):
    k, lam, lamP, mu, muP = args
    t0 = time.perf_counter()
    with redirect_stdout(io.StringIO()):
        old = int(old_CalcSoc(*args))
    old_sec = time.perf_counter() - t0
    t0 = time.perf_counter()
    new = int(CalcSoc(*args))
    new_sec = time.perf_counter() - t0
    equal = old == new
    row = {"lambda": str(lam), "mu": str(mu), "k": k,
           "lambda_prime": str(lamP), "mu_prime": str(muP),
           "original": old, "optimized": new, "match": equal,
           "old_seconds": round(old_sec, 6), "new_seconds": round(new_sec, 6)}
    comparison_rows.append(row)
    if not equal:
        failures.append(row)
    stats = by_k[k]
    stats["tests"] += 1
    stats["passed"] += int(equal)
    stats["nonzero"] += int(old != 0)
    stats["old_sec"] += old_sec
    stats["new_sec"] += new_sec
    pstats = by_pair[(lam, mu)]
    pstats["tests"] += 1
    pstats["passed"] += int(equal)
    if PROGRESS_EVERY and (i % PROGRESS_EVERY == 0 or i == len(test_cases)):
        print(f"{i:,}/{len(test_cases):,} coefficients; mismatches {len(failures)}; "
              f"elapsed {time.perf_counter()-started:.1f}s")

# Independent 3-factor LR checks from valid source/intermediate/remaining degrees.
lr3_cases = []
for outer in source_parts:
    n = degree(outer)
    for a in range(n + 1):
        for b in range(n - a + 1):
            c = n - a - b
            for p in generate_partitions(a):
                for q in generate_partitions(b):
                    for r in generate_partitions(c):
                        lr3_cases.append((tuple(p), tuple(q), tuple(r), outer))
rng.shuffle(lr3_cases)
lr3_cases = lr3_cases[:MAX_LR3_TESTS]
lr3_failures = []
for args in lr3_cases:
    with redirect_stdout(io.StringIO()):
        old = int(old_lrcoef3(*args))
    new = int(lrcoef3(*args))
    if old != new:
        lr3_failures.append({"arguments": str(args), "original": old, "optimized": new})

elapsed = time.perf_counter() - started
if comparison_rows:
    with open("hoyt_regression_summary_by_k.csv", "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=["k", "tests", "passed", "failed", "nonzero", "original_seconds", "optimized_seconds"])
        wr.writeheader()
        for k in sorted(by_k):
            s = by_k[k]
            wr.writerow({"k": k, "tests": s["tests"], "passed": s["passed"],
                         "failed": s["tests"]-s["passed"], "nonzero": s["nonzero"],
                         "original_seconds": round(s["old_sec"], 5),
                         "optimized_seconds": round(s["new_sec"], 5)})
with open("hoyt_regression_mismatches.csv", "w", newline="", encoding="utf-8") as f:
    fields = list(comparison_rows[0]) if comparison_rows else ["lambda", "mu", "k", "lambda_prime", "mu_prime", "original", "optimized", "match", "old_seconds", "new_seconds"]
    wr = csv.DictWriter(f, fieldnames=fields)
    wr.writeheader()
    wr.writerows(failures)
if SAVE_ALL_CSV and comparison_rows:
    with open("hoyt_regression_all.csv", "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=list(comparison_rows[0]))
        wr.writeheader()
        wr.writerows(comparison_rows)
print("Finished numerical comparisons; summary and mismatch CSVs saved.")


250/2,797 coefficients; mismatches 0; elapsed 0.1s
500/2,797 coefficients; mismatches 0; elapsed 0.2s
750/2,797 coefficients; mismatches 0; elapsed 0.2s
1,000/2,797 coefficients; mismatches 0; elapsed 0.3s
1,250/2,797 coefficients; mismatches 0; elapsed 0.4s
1,500/2,797 coefficients; mismatches 0; elapsed 0.5s
1,750/2,797 coefficients; mismatches 0; elapsed 0.6s
2,000/2,797 coefficients; mismatches 0; elapsed 0.7s
2,250/2,797 coefficients; mismatches 0; elapsed 0.8s
2,500/2,797 coefficients; mismatches 0; elapsed 0.8s
2,750/2,797 coefficients; mismatches 0; elapsed 0.9s
2,797/2,797 coefficients; mismatches 0; elapsed 0.9s
Finished numerical comparisons; summary and mismatch CSVs saved.


In [24]:
# Compact black-and-white HTML report (no pandas or matplotlib).
total = len(comparison_rows)
passed = total - len(failures)
lr3_passed = len(lr3_cases) - len(lr3_failures)
old_total = builtins.sum(v["old_sec"] for v in by_k.values())
new_total = builtins.sum(v["new_sec"] for v in by_k.values())
speedup = (old_total / new_total) if new_total > 0 else float("inf")
speedup = float(speedup)
num_pairs = len(by_pair)
nonzero = builtins.sum(v["nonzero"] for v in by_k.values())

style = """
<style>
.hoyt-report {font: 14px Arial, sans-serif;color:#111;max-width:900px}
.hoyt-report h2 {font-size:21px;font-weight:650;margin:0 0 12px}
.hoyt-report .sub {font-size:12px;color:#555;margin:0 0 16px}
.hoyt-report .metrics {display:flex;flex-wrap:wrap;gap:10px;margin:10px 0 18px}
.hoyt-report .metric {border:1px solid #bbb;border-radius:7px;padding:13px 16px;min-width:130px;background:#fff}
.hoyt-report .metric b {font-size:23px;display:block;margin-bottom:5px}
.hoyt-report .metric span {font-size:12px;color:#444}
.hoyt-report table {width:100%;border-collapse:collapse;margin-top:12px;font-size:13px}
.hoyt-report th {background:#111;color:white;text-align:right;padding:10px 12px}
.hoyt-report th:first-child,.hoyt-report td:first-child {text-align:left}
.hoyt-report td {padding:9px 12px;border-bottom:1px solid #ddd;text-align:right}
.hoyt-report tr:nth-child(even) td {background:#f4f4f4}
.hoyt-report .foot {font-size:12px;color:#555;margin-top:12px}
</style>"""
pass_rate = (100.0 * int(passed) / int(total)) if total else 0.0
report = style + f"""<div class='hoyt-report'>
<h2>Littlewood–Richardson regression report</h2>
<p class='sub'>Original vs. optimized implementation · reproducible seed {SEED} · source degree ≤ {MAX_DEGREE}</p>
<div class='metrics'>
<div class='metric'><b>{passed:,}/{total:,}</b><span>CalcSoc comparisons passed ({pass_rate:.2f}%)</span></div>
<div class='metric'><b>{len(failures):,}</b><span>CalcSoc mismatches</span></div>
<div class='metric'><b>{lr3_passed:,}/{len(lr3_cases):,}</b><span>lrcoef3 comparisons passed</span></div>
<div class='metric'><b>{num_pairs:,}</b><span>distinct source partition pairs</span></div>
<div class='metric'><b>{nonzero:,}</b><span>nonzero reference coefficients</span></div>
<div class='metric'><b>{speedup:.2f}×</b><span>measured coefficient-call speed ratio</span></div>
</div>
<h3>Breakdown by degree k</h3>
<table><thead><tr><th>k</th><th>Tested</th><th>Passed</th><th>Failed</th><th>Nonzero</th><th>Original (s)</th><th>Optimized (s)</th></tr></thead><tbody>"""
for k, s in sorted(by_k.items()):
    report += (f"<tr><td>{k}</td><td>{s['tests']:,}</td><td>{s['passed']:,}</td>"
               f"<td>{s['tests']-s['passed']:,}</td><td>{s['nonzero']:,}</td>"
               f"<td>{s['old_sec']:.3f}</td><td>{s['new_sec']:.3f}</td></tr>")
report += ("</tbody></table><p class='foot'>Timings include cache effects and are not a controlled speed benchmark. "
           "The regression checks agreement with the old code, not independent mathematical correctness. "
           "Files: hoyt_regression_summary_by_k.csv, hoyt_regression_mismatches.csv. "
           "Set SAVE_ALL_CSV=True for full per-case output.</p></div>")
display(HTML(report))
if failures or lr3_failures:
    print("First CalcSoc mismatches (up to 10):")
    for row in failures[:10]:
        print(row)
    print("First lrcoef3 mismatches (up to 10):")
    for row in lr3_failures[:10]:
        print(row)
else:
    print(f"PASS: all {total:,} CalcSoc and {len(lr3_cases):,} lrcoef3 comparisons matched.")
print("Cache statistics:", _lrcoeff_cached.cache_info(), lrcoef3_cached.cache_info())


k,Tested,Passed,Failed,Nonzero,Original (s),Optimized (s)
0,180,180,0,180,0.002,0.001
1,516,516,0,516,0.010,0.004
2,744,744,0,744,0.039,0.011
3,665,665,0,621,0.093,0.017
4,413,413,0,359,0.168,0.020
5,197,197,0,157,0.201,0.016
6,59,59,0,40,0.162,0.010
7,23,23,0,15,0.150,0.004


PASS: all 2,797 CalcSoc and 400 lrcoef3 comparisons matched.
Cache statistics: CacheInfo(hits=80420, misses=3748, maxsize=None, currsize=3748) CacheInfo(hits=485637, misses=17546, maxsize=None, currsize=17546)
